# Awash: dati aperti e primo turno di previsione

## tl;dr

Persistenza prevista prima dell'acquisizione del target: **2,9814 mm/giorno**.
Riferimento FAO AETI del 1–10 marzo 2024: **3,0417 mm/giorno**.
Errore assoluto: **0,0603 mm/giorno**. Un solo turno su stime riviste,
non abilità predittiva dimostrata o acqua irrigua misurata. Nessun nuovo fit.


## Context & Methods

Una prova dimostrativa vicino a Wonji nell'Awash, Etiopia, su un riquadro
WGS84 [39.20, 8.47, 39.22, 8.49], non un confine agricolo verificato.
Tre decadi di febbraio 2024; target 1–10 marzo 2024. La media giornaliera AETI
del target viene prevista ripetendo l'ultima media di febbraio.
Protocollo pubblicato in 819a076; storia e previsione in a3615e6 prima del target.

### Key Assumptions

- Il riferimento è una stima satellitare FAO; le revisioni attuali del 2024
  non ricostruiscono la disponibilità del dato durante il 2024.
- AETI include suolo, vegetazione e intercettazione. AETI meno pioggia non è
  acqua irrigata: falda, stoccaggio e altri flussi rimangono sconosciuti.
- Medie ponderate sulla sovrapposizione delle celle native con pesi sferici;
  la pioggia a circa 5 km non viene riscalata a 100 m.
- Richiediamo copertura valida completa; niente sostituzione dei mancanti con zero.
- Hash delle copie esatti, ricalcolo numerico entro 1e-10, nessuna azione operativa.

[Accesso FAO](https://www.fao.org/in-action/remote-sensing-for-water-productivity/wapor-data-access/en).
Si conserva il percorso dati → evidenza → esame scientifico e comunitario →
eventuale intervento con risultato verificato. I passaggi sul campo non sono stati eseguiti.


In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts/awash_irrigation.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
import awash_irrigation as awash
history = awash.load_phase('history')
target = awash.load_phase('target')
forecast = awash.verify_seal()
print('Protocol:', history['protocolHash'])
print('Historical source manifest:', history['manifestHash'])
print('Forecast file:', awash.digest((awash.EVIDENCE/'prediction.json').read_bytes()))


Protocol: 0xa45f599b48be4a3c3a81e52eceaab1c17d8569593a7679dd4bf4fa1a481ad3c7
Historical source manifest: 0x0c4927908e4c76f2c8a1e2413c40dbd52e82525e56370b602314c53c22383d1e
Forecast file: 0x9ff7604276dfa8a4e571da9deaba1514fc299165cbc31f3744160f839d4ac22a


## Data

Copie dei metadati e piccoli estratti nativi, non copie dei raster continentali.
Otto ritagli: AETI e pioggia su quattro periodi. Le sorgenti conservano versione,
scala 0,1, unità mm/giorno, aggiornamenti e generazione immutabile del file cloud.


In [2]:
print('Periodo / grandezza / media mm-giorno / celle valide / copertura')
for record in history['records'] + target['records']:
    assert record['complete'] and record['missingPixels'] == 0
    print(record['period']['code'], record['metric'], round(record['dailyMeanMm'], 4),
          f"{record['validPixels']}/{record['nativePixels']}", record['validAreaFraction'])
assert awash.dekad('2024-02-D3')['days'] == 9
print('Durata ultimo periodo febbraio:', awash.dekad('2024-02-D3')['days'], 'giorni')


Periodo / grandezza / media mm-giorno / celle valide / copertura
2024-02-D1 aeti 1.8825 462/462 1.0
2024-02-D1 precipitation 0.4 2/2 1.0
2024-02-D2 aeti 2.2023 462/462 1.0
2024-02-D2 precipitation 0.6 2/2 1.0
2024-02-D3 aeti 2.9814 462/462 1.0
2024-02-D3 precipitation 2.7999 2/2 1.0
2024-03-D1 aeti 3.0417 462/462 1.0
2024-03-D1 precipitation 7.9998 2/2 1.0
Durata ultimo periodo febbraio: 9 giorni


## Results

Verifica del riferimento e del punteggio contro le copie native.
Il valore successivo non entra in `prediction()`: usa soltanto `load_phase('history')`.


In [3]:
result = awash.evaluation()
saved = awash.read_json(awash.EVIDENCE/'result.json')
assert awash.same_value(result, saved)
print('Previsto:', result['predictedDailyMeanMm'], 'mm/giorno')
print('Stima FAO successiva:', result['actualEstimatedDailyMeanMm'], 'mm/giorno')
print('Errore assoluto:', result['absoluteErrorMmPerDay'], 'mm/giorno')
print('Errore su dieci giorni:', result['absoluteErrorMmPerDay'] * 10, 'mm')
print('Turni valutati:', result['scoredRounds'])


Previsto: 2.9813664208092177 mm/giorno
Stima FAO successiva: 3.041692839741762 mm/giorno
Errore assoluto: 0.060326418932544534 mm/giorno
Errore su dieci giorni: 0.6032641893254453 mm
Turni valutati: 1


In [4]:
assert result['irrigationWithdrawalM3'] is None
for field in ('skillEstablished', 'irrigationAttributionEstablished',
              'ecologicalImprovementVerified', 'fundingRelease', 'operationalEligible'):
    assert result[field] is False
print('Prelievo irriguo: sconosciuto. Abilità generale e beneficio ecologico: non dimostrati.')
print('Nessuna chiamata a LLM, fondi reali o attuatori.')


Prelievo irriguo: sconosciuto. Abilità generale e beneficio ecologico: non dimostrati.
Nessuna chiamata a LLM, fondi reali o attuatori.


## Takeaways

Il percorso informatico è riproducibile e il primo turno della persistenza è
vicino al riferimento FAO. La pioggia e AETI hanno griglie molto diverse;
nessun dato mancante viene interpolato. Un miglioramento scientifico richiederebbe
più periodi scelti prima della verifica, confronto con questa base e controllo
della disponibilità storica. Per un test dell'irrigazione reale servono confini
agricoli verificati e prelievi/volumi applicati misurati o esplicitamente stimati
con partner locali. Non estendiamo qui il modello del Willamette.
